# Generate Dataset

In [ ]:
!pip install transformers torch datasets pandas tqdm scikit-learn rouge-score accelerate -q

In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import nltk
from nltk.tokenize import sent_tokenize
import torch
from tqdm import tqdm
import glob
import json

nltk.download('punkt', quiet=True)

True

In [ ]:
original_dataset = load_dataset("Glepka/kinopoisk_classification")
print(f"Всего отзывов в исходном датасете: {len(original_dataset['train'])}")
print(f"Индексы: от 0 до {len(original_dataset['train']) - 1}")

ЗАГРУЗКА ИСХОДНОГО ДАТАСЕТА
Всего отзывов в исходном датасете: 57000
Индексы: от 0 до 56999


In [ ]:
print("\n" + "="*60)
print("ЗАГРУЗКА ФАЙЛОВ С РЕЗЮМЕ")
print("="*60)

all_files = glob.glob('*.csv')
print(f"Найдено файлов: {len(all_files)}")

all_summaries = {}

for file in all_files:
    try:
        df = pd.read_csv(file, encoding='utf-8')
        print(f"\nФайл: {file}")
        print(f"  Колонки: {df.columns.tolist()}")
        print(f"  Первые 3 индекса: {df['index'].head(3).tolist() if 'index' in df.columns else 'нет колонки index'}")

        if 'index' in df.columns and 'summary' in df.columns:
            for _, row in df.iterrows():
                idx = row['index']
                summary = row['summary']
                if pd.notna(summary) and isinstance(summary, str) and len(summary) > 20:
                    all_summaries[idx] = summary
        elif 'summary' in df.columns:
            for i, row in df.iterrows():
                summary = row['summary']
                if pd.notna(summary) and isinstance(summary, str) and len(summary) > 20:
                    all_summaries[i] = summary

    except Exception as e:
        print(f"Ошибка загрузки {file}: {e}")

print(f"\nВсего загружено резюме: {len(all_summaries)}")
print(f"Диапазон индексов: от {min(all_summaries.keys())} до {max(all_summaries.keys())}")
print(f"Пример индекса: {list(all_summaries.keys())[:10]}")


ЗАГРУЗКА ФАЙЛОВ С РЕЗЮМЕ
Найдено файлов: 14

Файл: kinopoisk_summaries_16000_to_17499.csv
  Колонки: ['index', 'summary', 'original_review_length']
  Первые 3 индекса: [16000, 16001, 16002]

Файл: kinopoisk_summaries_4500_to_5499.csv
  Колонки: ['index', 'summary', 'original_review_length']
  Первые 3 индекса: [4500, 4501, 4502]

Файл: kinopoisk_summaries_20450_to_21949.csv
  Колонки: ['index', 'summary', 'original_review_length']
  Первые 3 индекса: [20450, 20451, 20452]

Файл: kinopoisk_summaries_11000_to_12999.csv
  Колонки: ['index', 'summary', 'original_review_length']
  Первые 3 индекса: [11000, 11001, 11002]

Файл: kinopoisk_summaries_6500_to_7999.csv
  Колонки: ['index', 'summary', 'original_review_length']
  Первые 3 индекса: [6500, 6501, 6502]

Файл: kinopoisk_summaries_13000_to_14499.csv
  Колонки: ['index', 'summary', 'original_review_length']
  Первые 3 индекса: [13000, 13001, 13002]

Файл: kinopoisk_summaries_19450_to_20949.csv
  Колонки: ['index', 'summary', 'original_r

In [ ]:
print("\n" + "="*60)
print("АНАЛИЗ ПОКРЫТИЯ")
print("="*60)

original_indices = set(range(len(original_dataset['train'])))
summarized_indices = set(all_summaries.keys())
covered_indices = original_indices & summarized_indices

print(f"Всего индексов в исходном датасете: {len(original_indices)}")
print(f"Индексов с резюме: {len(summarized_indices)}")
print(f"Пересечение (есть и отзыв и резюме): {len(covered_indices)}")
print(f"Покрытие: {len(covered_indices)/len(original_indices)*100:.1f}%")

# Покажем распределение
print(f"\nПримеры покрытых индексов: {sorted(list(covered_indices))[:20]}")


АНАЛИЗ ПОКРЫТИЯ
Всего индексов в исходном датасете: 57000
Индексов с резюме: 20450
Пересечение (есть и отзыв и резюме): 20450
Покрытие: 35.9%

Примеры покрытых индексов: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [ ]:
print("\n" + "="*60)
print("СОЗДАНИЕ ФИНАЛЬНОГО ДАТАСЕТА")
print("="*60)

merged_data = []
missing_indices = []

for idx in sorted(covered_indices):
    try:
        review = original_dataset['train'][idx]['text']
        summary = all_summaries[idx]
        sentiment = original_dataset['train'][idx]['sentiment_text']

        # Базовая очистка
        if len(review) > 100 and len(summary) > 20:
            merged_data.append({
                'index': idx,
                'review': review,
                'summary': summary,
                'sentiment': sentiment
            })
        else:
            missing_indices.append(idx)
    except Exception as e:
        missing_indices.append(idx)

final_df = pd.DataFrame(merged_data)
print(f"Создано {len(final_df)} пар отзыв-резюме")
print(f"Пропущено (короткие тексты): {len(missing_indices)}")

print(f"\nПример первой пары:")
print(f"Индекс: {final_df['index'].iloc[0]}")
print(f"Отзыв: {final_df['review'].iloc[0][:150]}...")
print(f"Резюме: {final_df['summary'].iloc[0][:150]}...")
print(f"Тональность: {final_df['sentiment'].iloc[0]}")


СОЗДАНИЕ ФИНАЛЬНОГО ДАТАСЕТА
Создано 20450 пар отзыв-резюме
Пропущено (короткие тексты): 0

Пример первой пары:
Индекс: 0
Отзыв: Сегодня в очередной раз пересмотрел один из моих любимых фильмов, и вдруг захотелось написать о нем отзыв. Этот фильм должен быть оценен в первую очер...
Резюме: **Главная мысль:** Оценка фильма Скоттом Эдкинсом (Юрий Бойко) основана преимущественно на качестве боевых сцен.
**Плюсы:** 
1. Боевые сцены Скотта Эд...
Тональность: pos


## Dataset Statistics

In [ ]:
print(f"Всего пар: {len(final_df)}")
print(f"Средняя длина отзыва: {final_df['review'].str.len().mean():.0f} символов")
print(f"Средняя длина резюме: {final_df['summary'].str.len().mean():.0f} символов")

print("\nРаспределение тональности:")
print(final_df['sentiment'].value_counts())

print("\nРаспределение индексов:")
print(f"Минимальный индекс: {final_df['index'].min()}")
print(f"Максимальный индекс: {final_df['index'].max()}")
print(f"Шаг между индексами: нерегулярный (пропуски есть)")


СТАТИСТИКА ДАТАСЕТА
Всего пар: 20450
Средняя длина отзыва: 2266 символов
Средняя длина резюме: 202 символов

Распределение тональности:
sentiment
neg    6912
neu    6786
pos    6752
Name: count, dtype: int64

Распределение индексов:
Минимальный индекс: 0
Максимальный индекс: 21949
Шаг между индексами: нерегулярный (пропуски есть)


# Baseline (TF-IDF)

In [ ]:
class TFIDFBaseline:
    def __init__(self, num_sentences=3):
        self.num_sentences = num_sentences
        self.vectorizer = TfidfVectorizer(min_df=1)

    def summarize(self, text):
        if not isinstance(text, str) or len(text) < 50:
            return text if isinstance(text, str) else ""

        try:
            sentences = sent_tokenize(text, language='russian')
        except:
            sentences = [s.strip() for s in text.replace('!', '.').replace('?', '.').split('.') if len(s.strip()) > 10]

        if len(sentences) <= self.num_sentences:
            return text

        try:
            tfidf_matrix = self.vectorizer.fit_transform(sentences)
            sentence_scores = tfidf_matrix.sum(axis=1).A1
            top_indices = sentence_scores.argsort()[-self.num_sentences:][::-1]
            top_indices = sorted(top_indices)
            return ' '.join([sentences[i] for i in top_indices])
        except:
            return ' '.join(sentences[:self.num_sentences])

baseline = TFIDFBaseline()
test_idx = final_df['index'].iloc[0]
test_review = final_df['review'].iloc[0]
print(f"Индекс {test_idx}:")
print(f"Baseline: {baseline.summarize(test_review)[:150]}...")
print(f"Эталон:   {final_df['summary'].iloc[0][:150]}...")


BASELINE МОДЕЛЬ
Индекс 0:
Baseline: Но это, конечно, не его вина, а в первую очередь оператора, во вторую режиссера - первый снял с неверного ракурса, второй недоглядел при обработке. Эл...
Эталон:   **Главная мысль:** Оценка фильма Скоттом Эдкинсом (Юрий Бойко) основана преимущественно на качестве боевых сцен.
**Плюсы:** 
1. Боевые сцены Скотта Эд...


## Authorization

In [ ]:
!pip install transformers torch datasets pandas tqdm scikit-learn rouge-score accelerate huggingface_hub -q

In [1]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from huggingface_hub import login, HfApi
import nltk
from nltk.tokenize import sent_tokenize
import torch
from tqdm import tqdm
import glob
import os

nltk.download('punkt', quiet=True)

# Вход в Hugging Face (потребуется ваш токен)
print("="*60)
print("АВТОРИЗАЦИЯ НА HUGGING FACE HUB")
print("="*60)
print("Получите токен: https://huggingface.co/settings/tokens")
hf_token = input("Введите ваш HF токен: ").strip()
login(token=hf_token)
print("✅ Авторизация успешна!")

KeyboardInterrupt: 

## Preparing Dataset

In [ ]:
train_df, temp_df = train_test_split(final_df[['review', 'summary']], test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

model_name = "sberbank-ai/ruT5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

max_input_len = 512
max_target_len = 128

def preprocess_function(examples):
    inputs = [f"summarize: {text}" for text in examples["review"]]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_len,
        truncation=True,
        padding="max_length"
    )

    targets = [text for text in examples["summary"]]
    labels = tokenizer(
        targets,
        max_length=max_target_len,
        truncation=True,
        padding="max_length"
    )

    labels_ids = []
    for seq in labels["input_ids"]:
        labels_ids.append([-100 if t == tokenizer.pad_token_id else t for t in seq])

    model_inputs["labels"] = labels_ids
    return model_inputs

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

print("Токенизация...")
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)
test_dataset = test_dataset.map(preprocess_function, batched=True, remove_columns=test_dataset.column_names)

datasets = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

print("✅ Данные готовы")


ПОДГОТОВКА ДАННЫХ
Train: 16360
Validation: 2045
Test: 2045
Токенизация...


Map:   0%|          | 0/16360 [00:00<?, ? examples/s]

Map:   0%|          | 0/2045 [00:00<?, ? examples/s]

Map:   0%|          | 0/2045 [00:00<?, ? examples/s]

✅ Данные готовы


## Training RuT5-BASE (Previous Version)

In [ ]:
print("\n" + "="*60)
print("ОБУЧЕНИЕ RUT5-BASE")
print("="*60)

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def safe_rouge_metrics(eval_pred):
    """Безопасная версия с ROUGE и отладочной информацией"""
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    try:
        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    except Exception as e:
        print(f"Ошибка декодирования: {e}")
        return {"rouge1": 0, "rouge2": 0, "rougeL": 0}

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    rouge_scores = {
        'rouge1': [],
        'rouge2': [],
        'rougeL': [],
    }

    for pred, label in zip(decoded_preds, decoded_labels):
        if not pred or not label:
            continue

        try:
            scores = scorer.score(label, pred)
            for key in rouge_scores:
                rouge_scores[key].append(scores[key].fmeasure)
        except Exception as e:
            print(f"Ошибка при вычислении ROUGE: {e}")
            continue

    result = {
        'rouge1': np.mean(rouge_scores['rouge1']) if rouge_scores['rouge1'] else 0,
        'rouge2': np.mean(rouge_scores['rouge2']) if rouge_scores['rouge2'] else 0,
        'rougeL': np.mean(rouge_scores['rougeL']) if rouge_scores['rougeL'] else 0,
        'sample_pred': decoded_preds[0][:100] if decoded_preds else "N/A",
        'sample_label': decoded_labels[0][:100] if decoded_labels else "N/A"
    }

    return result


training_args = Seq2SeqTrainingArguments(
    output_dir="./rut5-summarization",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    report_to="none",
    logging_steps=50,
    push_to_hub=True,
    hub_model_id="Auttar/RuT5SentimentSummarization",
    hub_strategy="every_save",
    # Add these to prevent overflow issues
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
)


# Загрузка модели
model = T5ForConditionalGeneration.from_pretrained(model_name)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Trainer (БЕЗ tokenizer аргумента)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    data_collator=data_collator,
    compute_metrics=safe_rouge_metrics,
)

print("🚀 Начало обучения...")
print("✅ После обучения модель автоматически загрузится на Hugging Face Hub")
print("📍 Репозиторий: Auttar/RuT5SentimentSummarization")

# Обучение
trainer.train()

# Финальное сохранение и пуш на HF
print("\n💾 Сохранение модели на Hugging Face Hub...")
trainer.push_to_hub(commit_message="Final model for sentiment summarization")


ОБУЧЕНИЕ RUT5-BASE


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


🚀 Начало обучения...
✅ После обучения модель автоматически загрузится на Hugging Face Hub
📍 Репозиторий: Auttar/RuT5SentimentSummarization


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1,1.931485,1.669748,0,0,0
2,1.728364,1.589400,0,0,0
3,1.583212,1.564702,0,0,0


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ization/training_args.bin: 100%|##########| 5.39kB / 5.39kB            

  ...ization/model.safetensors:  14%|#4        |  168MB / 1.19GB            

CommitInfo(commit_url='https://huggingface.co/Auttar/RuT5SentimentSummarization/commit/0c1fee1c5b24eef9d653a823a264449c5e313788', commit_message='Final model for sentiment summarization', commit_description='', oid='0c1fee1c5b24eef9d653a823a264449c5e313788', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Auttar/RuT5SentimentSummarization', endpoint='https://huggingface.co', repo_type='model', repo_id='Auttar/RuT5SentimentSummarization'), pr_revision=None, pr_num=None)

## Eval And Save (Previous Version)

In [ ]:
baseline_scores = []
for i in range(min(100, len(test_df))):
    pred = baseline.summarize(test_df.iloc[i]['review'])
    ref = test_df.iloc[i]['summary']
    if pred and ref:
        scores = scorer.score(ref, pred)
        baseline_scores.append(scores)

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

rut5_scores = []
for i in tqdm(range(min(100, len(test_df))), desc="Оценка ruT5"):
    input_text = f"summarize: {test_df.iloc[i]['review']}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(inputs.input_ids, max_length=128, num_beams=4)

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    ref = test_df.iloc[i]['summary']

    if pred and ref:
        scores = scorer.score(ref, pred)
        rut5_scores.append(scores)

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ")
print("="*60)

if baseline_scores:
    print(f"\nBaseline (TF-IDF):")
    print(f"  ROUGE-1: {np.mean([s['rouge1'].fmeasure for s in baseline_scores]):.4f}")
    print(f"  ROUGE-2: {np.mean([s['rouge2'].fmeasure for s in baseline_scores]):.4f}")
    print(f"  ROUGE-L: {np.mean([s['rougeL'].fmeasure for s in baseline_scores]):.4f}")

if rut5_scores:
    print(f"\nruT5-base:")
    print(f"  ROUGE-1: {np.mean([s['rouge1'].fmeasure for s in rut5_scores]):.4f}")
    print(f"  ROUGE-2: {np.mean([s['rouge2'].fmeasure for s in rut5_scores]):.4f}")
    print(f"  ROUGE-L: {np.mean([s['rougeL'].fmeasure for s in rut5_scores]):.4f}")

metrics_df = pd.DataFrame([
    {'model': 'Baseline (TF-IDF)',
     'rouge1': np.mean([s['rouge1'].fmeasure for s in baseline_scores]),
     'rouge2': np.mean([s['rouge2'].fmeasure for s in baseline_scores]),
     'rougeL': np.mean([s['rougeL'].fmeasure for s in baseline_scores])},
    {'model': 'ruT5-base',
     'rouge1': np.mean([s['rouge1'].fmeasure for s in rut5_scores]),
     'rouge2': np.mean([s['rouge2'].fmeasure for s in rut5_scores]),
     'rougeL': np.mean([s['rougeL'].fmeasure for s in rut5_scores])}
])

metrics_df.to_csv("evaluation_results.csv", index=False)




ОЦЕНКА МОДЕЛЕЙ


Оценка ruT5: 100%|██████████| 100/100 [03:12<00:00,  1.93s/it]


РЕЗУЛЬТАТЫ

Baseline (TF-IDF):
  ROUGE-1: 0.0331
  ROUGE-2: 0.0029
  ROUGE-L: 0.0291

ruT5-base:
  ROUGE-1: 0.1337
  ROUGE-2: 0.0673
  ROUGE-L: 0.1337


NameError: name 'api' is not defined

## Save

In [ ]:
api = HfApi()

# Загружаем метрики на HF
api.upload_file(
    path_or_fileobj="evaluation_results.csv",
    path_in_repo="evaluation_results.csv",
    repo_id="Auttar/RuT5SentimentSummarization",
    token=hf_token
)

print("\n✅ Результаты сохранены в репозитории!")


✅ Результаты сохранены в репозитории!


## model card (to HF, Previous Version)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Model card
model_card = """---
language: ru
tags:
- summarization
- t5
- russian
- sentiment-analysis
license: mit
---

# RuT5-Base for Sentiment Summarization

## Описание
Модель для генеративной суммаризации отзывов на русском языке. Обучалась на датасете Kinopoisk с использованием дистилляции знаний от LLM.

## Задача
Создание кратких, информативных резюме отзывов с выделением ключевых плюсов и минусов.

## Метрики на тестовой выборке
(Будут добавлены после завершения обучения)

## Использование
```python
from transformers import T5ForConditionalGeneration, T5Tokenizer

model = T5ForConditionalGeneration.from_pretrained("Auttar/RuT5SentimentSummarization")
tokenizer = T5Tokenizer.from_pretrained("Auttar/RuT5SentimentSummarization")

def summarize(review):
    input_text = f"summarize: {review}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(inputs.input_ids, max_length=128, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Пример
review = "Фильм отличный! Актеры играют великолепно, сюжет захватывает."
print(summarize(review))

# Обучение
- Базовая модель: sberbank-ai/ruT5-base

- Датасет: Kinopoisk с сгенерированными резюме

- Эпохи: 3

- Learning rate: 5e-5

- Batch size: 4

"""


ДОБАВЛЕНИЕ MODEL CARD И МЕТРИК


In [ ]:
with open("./rut5-summarization/README.md", "w", encoding='utf-8') as f:
  f.write(updated_readme)

In [ ]:
try:
  api.upload_file(
  path_or_fileobj="./rut5-summarization/README.md",
  path_in_repo="README.md",
  repo_id="Auttar/RuT5SentimentSummarization",
  token=hf_token
  )
  print("✅ Model card загружен")
except Exception as e:
  print(f"Ошибка загрузки model card: {e}")

✅ Model card загружен


In [ ]:
baseline_scores = []
for i in range(min(100, len(test_df))):
    pred = baseline.summarize(test_df.iloc[i]['review'])
    ref = test_df.iloc[i]['summary']
    if pred and ref:
        scores = scorer.score(ref, pred)
        baseline_scores.append(scores)

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

rut5_scores = []
for i in tqdm(range(min(100, len(test_df))), desc="Оценка ruT5"):
    input_text = f"summarize: {test_df.iloc[i]['review']}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(inputs.input_ids, max_length=128, num_beams=4)

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    ref = test_df.iloc[i]['summary']

    if pred and ref:
        scores = scorer.score(ref, pred)
        rut5_scores.append(scores)

baseline_metrics = {}
rut5_metrics = {}

if baseline_scores:
    baseline_metrics = {
        'rouge1': np.mean([s['rouge1'].fmeasure for s in baseline_scores]),
        'rouge2': np.mean([s['rouge2'].fmeasure for s in baseline_scores]),
        'rougeL': np.mean([s['rougeL'].fmeasure for s in baseline_scores])
    }
    print(f"\nBaseline (TF-IDF):")
    print(f"  ROUGE-1: {baseline_metrics['rouge1']:.4f}")
    print(f"  ROUGE-2: {baseline_metrics['rouge2']:.4f}")
    print(f"  ROUGE-L: {baseline_metrics['rougeL']:.4f}")

if rut5_scores:
    rut5_metrics = {
        'rouge1': np.mean([s['rouge1'].fmeasure for s in rut5_scores]),
        'rouge2': np.mean([s['rouge2'].fmeasure for s in rut5_scores]),
        'rougeL': np.mean([s['rougeL'].fmeasure for s in rut5_scores])
    }
    print(f"\nruT5-base:")
    print(f"  ROUGE-1: {rut5_metrics['rouge1']:.4f}")
    print(f"  ROUGE-2: {rut5_metrics['rouge2']:.4f}")
    print(f"  ROUGE-L: {rut5_metrics['rougeL']:.4f}")

# Update README with metrix
if baseline_metrics and rut5_metrics:
    updated_readme = f"""---
language: ru
tags:
- summarization
- t5
- russian
- sentiment-analysis
license: mit
---

# RuT5-Base for Sentiment Summarization

## Описание
Модель для генеративной суммаризации отзывов на русском языке. Обучалась на датасете Kinopoisk с использованием дистилляции знаний от LLM.

## Задача
Создание кратких, информативных резюме отзывов с выделением ключевых плюсов и минусов.

## Метрики на тестовой выборке
| Модель | ROUGE-1 | ROUGE-2 | ROUGE-L |
|--------|---------|---------|---------|
| Baseline (TF-IDF) | {baseline_metrics['rouge1']:.4f} | {baseline_metrics['rouge2']:.4f} | {baseline_metrics['rougeL']:.4f} |
| **ruT5-base** | **{rut5_metrics['rouge1']:.4f}** | **{rut5_metrics['rouge2']:.4f}** | **{rut5_metrics['rougeL']:.4f}** |

## Использование
```python
from transformers import T5ForConditionalGeneration, T5Tokenizer

model = T5ForConditionalGeneration.from_pretrained("Auttar/RuT5SentimentSummarization")
tokenizer = T5Tokenizer.from_pretrained("Auttar/RuT5SentimentSummarization")

def summarize(review):
    input_text = f"summarize: {{review}}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(inputs.input_ids, max_length=128, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Пример
review = "Фильм отличный! Актеры играют великолепно, сюжет захватывает."
print(summarize(review))

# Обучение
- Базовая модель: sberbank-ai/ruT5-base

- Датасет: Kinopoisk с сгенерированными резюме

- Эпохи: 3

- Learning rate: 5e-5

- Batch size: 4

"""



ОЦЕНКА МОДЕЛЕЙ


Оценка ruT5: 100%|██████████| 100/100 [03:31<00:00,  2.11s/it]


РЕЗУЛЬТАТЫ

Baseline (TF-IDF):
  ROUGE-1: 0.0331
  ROUGE-2: 0.0029
  ROUGE-L: 0.0291

ruT5-base:
  ROUGE-1: 0.1337
  ROUGE-2: 0.0673
  ROUGE-L: 0.1337


## Inference (previous version)

In [ ]:
print("Загрузка модели с Hugging Face Hub...")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

test_model = T5ForConditionalGeneration.from_pretrained("Auttar/RuT5SentimentSummarization")
test_tokenizer = T5Tokenizer.from_pretrained("Auttar/RuT5SentimentSummarization")
test_model.to(device)
test_model.eval()

test_reviews = [
    "Фильм просто бомба! Актеры играют отлично, сюжет держит в напряжении до конца. Операторская работа на высоте. Очень рекомендую!",
    "Полное разочарование. Скучный сюжет, слабая игра актеров, затянутые сцены. Деньги на ветер. Не советую тратить время.",
    "Неплохой фильм на один раз. Есть смешные моменты, но в целом ничего выдающегося. Среднячок, можно посмотреть если делать нечего."
]

print("\nРезультаты:")
for i, review in enumerate(test_reviews, 1):
    input_text = f"summarize: {review}"
    inputs = test_tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)

    with torch.no_grad():
        outputs = test_model.generate(
            inputs.input_ids,
            max_length=128,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    summary = test_tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"\n--- Пример {i} ---")
    print(f"Отзыв: {review[:100]}...")
    print(f"Резюме: {summary}")

print("\n✅ Модель работает корректно!")
print("📍 Модель доступна по ссылке: https://huggingface.co/Auttar/RuT5SentimentSummarization")


ТЕСТОВЫЙ ИНФЕРЕНС
Загрузка модели с Hugging Face Hub...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Результаты:

--- Пример 1 ---
Отзыв: Фильм просто бомба! Актеры играют отлично, сюжет держит в напряжении до конца. Операторская работа н...
Резюме: **Фильм "Бомба"**: Актёры играют отлично, сюжет держит в напряжении до конца. Операторская работа на высоком уровне, операторская работа отличная. **Главная мысль**: Фильм обладает отличными актерами и операторской...

--- Пример 2 ---
Отзыв: Полное разочарование. Скучный сюжет, слабая игра актеров, затянутые сцены. Деньги на ветер. Не совет...
Резюме: **Резюме:** **Главная мысль:** Отрицание качества сюжета и актерского мастерства. **Плюсы:** 1. Скучный сюжет 2. Затянутые сцены Минусы: 1. Недостаток сюжета Минус...

--- Пример 3 ---
Отзыв: Неплохой фильм на один раз. Есть смешные моменты, но в целом ничего выдающегося. Среднячок, можно по...
Резюме: **Резюме:** **Главная мысль:** Фильм имеет смешные моменты, но не выдающийся. **Плюсы:** 1. Смешные моменты. 2. Смеховые моменты. Минусы: 1. Недостаток смешных моментов. ...

✅ Модель работае

## Dataset to HF

In [ ]:
from huggingface_hub import HfApi
from datasets import Dataset, DatasetDict
import json
import os

os.makedirs("./kinopoisk_reviews_summarization", exist_ok=True)

train_df.to_json("./kinopoisk_reviews_summarization/train.json", orient="records", force_ascii=False, indent=2)
val_df.to_json("./kinopoisk_reviews_summarization/validation.json", orient="records", force_ascii=False, indent=2)
test_df.to_json("./kinopoisk_reviews_summarization/test.json", orient="records", force_ascii=False, indent=2)

dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

dataset_dict.save_to_disk("./kinopoisk_reviews_summarization/dataset")

dataset_readme = """---
language:
- ru
license: mit
---

# Kinopoisk Reviews Summarization Dataset

## Описание
Датасет для обучения моделей суммаризации отзывов на русском языке. Содержит отзывы о фильмах с сайта Kinopoisk и сгенерированные краткие резюме.

## Структура
- `train.json`: 80% данных (обучение)
- `validation.json`: 10% данных (валидация)
- `test.json`: 10% данных (тестирование)

## Формат данных
Каждый пример содержит:
- `review`: Полный текст отзыва (строка)
- `summary`: Краткое резюме отзыва (строка)

## Источники
1. **Исходные данные**: Отзывы с Kinopoisk (Glepka)
2. **Сгенерированные резюме**: Созданы с использованием дистилляции знаний от LLM
3. **Обработка**: Дополнены собственными примерами для балансировки

## Пример использования
```python
from datasets import load_dataset

dataset = load_dataset("Auttar/KinopoiskReviewsSummarization")

for example in dataset["train"].select(range(3)):
    print(f"Отзыв: {example['review'][:100]}...")
    print(f"Резюме: {example['summary']}\\n")
"""


ВЫГРУЗКА ДАТАСЕТА НА HUB


Saving the dataset (0/1 shards):   0%|          | 0/16360 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2045 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2045 [00:00<?, ? examples/s]

In [ ]:
with open("./kinopoisk_reviews_summarization/README.md", "w", encoding='utf-8') as f:
  f.write(dataset_readme)

In [ ]:
api.create_repo(
repo_id="Auttar/KinopoiskReviewsSummarization",
token=hf_token,
repo_type="dataset",
exist_ok=True
)

api.upload_file(
    path_or_fileobj="./kinopoisk_reviews_summarization/README.md",
    path_in_repo="README.md",
    repo_id="Auttar/KinopoiskReviewsSummarization",
    repo_type="dataset",
    token=hf_token
)

for split in ["train", "validation", "test"]:
    api.upload_file(
        path_or_fileobj=f"./kinopoisk_reviews_summarization/{split}.json",
        path_in_repo=f"{split}.json",
        repo_id="Auttar/KinopoiskReviewsSummarization",
        repo_type="dataset",
        token=hf_token
    )

api.upload_folder(
    folder_path="./kinopoisk_reviews_summarization/dataset",
    path_in_repo="dataset",
    repo_id="Auttar/KinopoiskReviewsSummarization",
    repo_type="dataset",
    token=hf_token
)

print("✅ Датасет успешно загружен!")
print("📍 https://huggingface.co/datasets/Auttar/KinopoiskReviewsSummarization")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._summarization/train.json:   1%|1         | 1.01MB / 74.1MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...data-00000-of-00001.arrow:   1%|1         |  997kB / 73.4MB            

  ...data-00000-of-00001.arrow:  26%|##6       | 2.38MB / 9.06MB            

  ...data-00000-of-00001.arrow:  26%|##6       | 2.40MB / 9.14MB            

✅ Датасет успешно загружен!
📍 https://huggingface.co/datasets/Auttar/KinopoiskReviewsSummarization


## Quality Inference

In [ ]:
print("\n" + "="*80)
print("КАЧЕСТВЕННОЕ СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*80)

def print_comparison(review, baseline_summary, rut5_summary):
    print("\n" + "-" * 80)
    print(f"ОТЗЫВ: {review[:150]}...")
    print("\n" + "-"*80)
    print(f"BASELINE (TF-IDF) СУММАРИЗАЦИЯ:")
    print(f"{baseline_summary}\n")
    print("-"*80)
    print(f"RuT5 СУММАРИЗАЦИЯ:")
    print(f"{rut5_summary}\n")
    print("-"*80)
print("🔄 Инициализация моделей...")

model_rut5 = T5ForConditionalGeneration.from_pretrained("Auttar/RuT5SentimentSummarization")
tokenizer_rut5 = T5Tokenizer.from_pretrained("Auttar/RuT5SentimentSummarization")
model_rut5.to(device)
model_rut5.eval()

test_cases = [
    {
        "review": "Фильм просто бомба! Актеры играют отлично, сюжет держит в напряжении до конца. " +
                 "Операторская работа на высоте, особенно понравились сцены ночного города. " +
                 "Саундтрек идеально дополняет атмосферу. Очень рекомендую всем любителям триллеров!",
        "context": "Позитивный отзыв о триллере"
    },
    {
        "review": "Полное разочарование. Сюжет предсказуем как в дешевом сериале, " +
                 "актеры переигрывают, а спецэффекты выглядят как из 90-х. " +
                 "Режиссер явно не понимал, что хочет сказать зрителю. " +
                 "Единственный плюс - короткое время просмотра (1.5 часа).",
        "context": "Негативный отзыв о фильме"
    },
    {
        "review": "Неплохой фильм на один раз. Есть несколько смешных моментов, " +
                 "хотя юмор местами натянут. Главный герой харизматичен, но " +
                 "вторичные персонажи абсолютно стереотипны. " +
                 "Среднячок, можно посмотреть если делать нечего, но " +
                 "ожидать чего-то выдающегося не стоит.",
        "context": "Нейтральный отзыв о комедии"
    },
    {
        "review": "Отличная экранная адаптация книги! Сценаристы сумели сохранить " +
                 "атмосферу оригинала, при этом добавив интересные визуальные решения. " +
                 "Главная героиня в исполнении Ангелины Джоли просто великолепна. " +
                 "Единственный минус - несколько затянутые сцены с пейзажами. " +
                 "В целом, фильм обязателен к просмотру для фанатовантастики.",
        "context": "Позитивный отзыв об экранизации"
    },
    {
        "review": "Фильм про войну, который не о войне. Вместо боевых сцен " +
                 "мы видим историю дружбы и выживания в экстремальных условиях. " +
                 "Актерская игра на высоте, особенно понравился главный герой. " +
                 "Хотя тем, кто ожидает динамичного экшена, может быть скучно.",
        "context": "Аналитический отзыв о военной драме"
    }
]

print(f"\n✅ Загружено {len(test_cases)} тестовых случаев")
print("🔍 Начало сравнительного анализа...")

for i, case in enumerate(test_cases, 1):
    print(f"\n\n{'='*80}")
    print(f"СЛУЧАЙ {i}: {case['context']}")
    print(f"{'='*80}")

    # Baseline суммаризация
    baseline_summary = baseline.summarize(case["review"])

    # RuT5 суммаризация
    input_text = f"summarize: {case['review']}"
    inputs = tokenizer_rut5(input_text, return_tensors="pt",
                          max_length=512, truncation=True).to(device)

    with torch.no_grad():
        outputs = model_rut5.generate(
            inputs.input_ids,
            max_length=128,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=1.2
        )

    rut5_summary = tokenizer_rut5.decode(outputs[0], skip_special_tokens=True)

    print_comparison(case["review"], baseline_summary, rut5_summary)



КАЧЕСТВЕННОЕ СРАВНЕНИЕ МОДЕЛЕЙ
🔄 Инициализация моделей...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



✅ Загружено 5 тестовых случаев
🔍 Начало сравнительного анализа...


СЛУЧАЙ 1: Позитивный отзыв о триллере

--------------------------------------------------------------------------------
ОТЗЫВ: Фильм просто бомба! Актеры играют отлично, сюжет держит в напряжении до конца. Операторская работа на высоте, особенно понравились сцены ночного город...

--------------------------------------------------------------------------------
BASELINE (TF-IDF) СУММАРИЗАЦИЯ:
Актеры играют отлично, сюжет держит в напряжении до конца. Операторская работа на высоте, особенно понравились сцены ночного города. Очень рекомендую всем любителям триллеров!

--------------------------------------------------------------------------------
RuT5 СУММАРИЗАЦИЯ:
**Фильм "Просто бомба"**: Актёры играют отлично, операторская работа на высоте. Саундтрек и саундтрек отлично дополняют атмосферу. **Главная мысль**: Фильм обладает отличными актерами и операторской...

--------------------------------------------------------

# New Models And Semantic Metrix

In [ ]:
!pip install transformers torch datasets pandas tqdm scikit-learn rouge-score evaluate accelerate sentencepiece huggingface_hub sentence-transformers -q
!pip install git+https://github.com/google-research/bleurt.git -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
!pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    T5ForConditionalGeneration, T5Tokenizer,
    BartForConditionalGeneration, BartTokenizer,
    AutoModelForSeq2SeqLM, AutoTokenizer,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from huggingface_hub import login, HfApi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from bert_score import BERTScorer
from sentence_transformers import SentenceTransformer
import nltk
from nltk.tokenize import sent_tokenize
import torch
from tqdm import tqdm
import json
import time
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ Все библиотеки загружены!")

✅ Все библиотеки загружены!


## Download Dataset

In [ ]:
dataset = load_dataset("Auttar/KinopoiskReviewsSummarization")

print(f"Train: {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"Test: {len(dataset['test'])}")

print("\nПример из датасета:")
print(f"Отзыв: {dataset['train'][0]['review'][:200]}...")
print(f"Резюме: {dataset['train'][0]['summary'][:200]}...")

Загрузка датасета...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/38.4M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/4.79M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/4.76M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16360 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2045 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2045 [00:00<?, ? examples/s]

Train: 16360
Validation: 2045
Test: 2045

Пример из датасета:
Отзыв: Мне первая часть «Не вижу зла» нравится до сих пор. Уж больно необычный был там маньяк, и он довольно круто расправлялся со своими жертвами. Фильм как-то выделялся из большинства слэшеров, ну а финаль...
Резюме: **Резюме:**  
**Главная мысль:** Отличительной чертой первого фильма является уникальный персонаж маньяка и оригинальное поведение автора. Следующий фильм оказывается менее интересным и утомительным д...


## Metrix Scorer

In [ ]:
class FullMetricsCalculator:
    """Расчет всех метрик: ROUGE, BERTScore, SBERT Semantic Similarity"""

    def __init__(self):
        self.rouge_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        self.sbert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
        self.bert_scorer = BERTScorer(
            lang="ru",
            rescale_with_baseline=False,
            device='cuda' if torch.cuda.is_available() else 'cpu'
        )

        print("✅ Все метрики инициализированы")

    def calculate_rouge(self, predictions, references):
        rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}

        for pred, ref in zip(predictions, references):
            if pred and ref:
                try:
                    scores = self.rouge_scorer.score(ref, pred)
                    rouge_scores['rouge1'].append(scores['rouge1'].fmeasure)
                    rouge_scores['rouge2'].append(scores['rouge2'].fmeasure)
                    rouge_scores['rougeL'].append(scores['rougeL'].fmeasure)
                except:
                    continue

        return {
            'rouge1': np.mean(rouge_scores['rouge1']) if rouge_scores['rouge1'] else 0,
            'rouge2': np.mean(rouge_scores['rouge2']) if rouge_scores['rouge2'] else 0,
            'rougeL': np.mean(rouge_scores['rougeL']) if rouge_scores['rougeL'] else 0
        }

    def calculate_bertscore(self, predictions, references):
        try:
            P, R, F1 = self.bert_scorer.score(predictions, references)
            return {
                'bert_score_precision': P.mean().item(),
                'bert_score_recall': R.mean().item(),
                'bert_score_f1': F1.mean().item()
            }
        except Exception as e:
            print(f"  BERTScore ошибка: {e}")
            return {
                'bert_score_precision': 0,
                'bert_score_recall': 0,
                'bert_score_f1': 0
            }

    def calculate_semantic_similarity(self, predictions, references):
        """Расчет семантической близости через SBERT"""
        pred_embeddings = self.sbert_model.encode(predictions, show_progress_bar=False)
        ref_embeddings = self.sbert_model.encode(references, show_progress_bar=False)

        similarities = []
        for pred_emb, ref_emb in zip(pred_embeddings, ref_embeddings):
            sim = cosine_similarity([pred_emb], [ref_emb])[0][0]
            similarities.append(sim)

        return {
            'sbert_similarity_mean': np.mean(similarities),
            'sbert_similarity_std': np.std(similarities),
            'sbert_similarity_min': np.min(similarities),
            'sbert_similarity_max': np.max(similarities)
        }

    def calculate_all(self, predictions, references, model_name="Model"):
        print(f"\n  Расчет метрик для {model_name}...")

        print("    ROUGE...")
        rouge = self.calculate_rouge(predictions, references)

        print("    BERTScore...")
        bertscore = self.calculate_bertscore(predictions, references)

        print("    SBERT Semantic Similarity...")
        semantic = self.calculate_semantic_similarity(predictions, references)

        all_metrics = {**rouge, **bertscore, **semantic}

        return all_metrics

# Инициализируем калькулятор метрик
metrics_calculator = FullMetricsCalculator()

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Все метрики инициализированы


## Update and Evaluate Baseline

In [ ]:
class TFIDFBaseline:
    def __init__(self, num_sentences=3):
        self.num_sentences = num_sentences
        self.vectorizer = TfidfVectorizer(min_df=1)

    def summarize(self, text):
        if not isinstance(text, str) or len(text) < 50:
            return text if isinstance(text, str) else ""

        try:
            sentences = sent_tokenize(text, language='russian')
        except:
            sentences = [s.strip() for s in text.replace('!', '.').replace('?', '.').split('.') if len(s.strip()) > 10]

        if len(sentences) <= self.num_sentences:
            return text

        try:
            tfidf_matrix = self.vectorizer.fit_transform(sentences)
            sentence_scores = tfidf_matrix.sum(axis=1).A1
            top_indices = sentence_scores.argsort()[-self.num_sentences:][::-1]
            top_indices = sorted(top_indices)
            return ' '.join([sentences[i] for i in top_indices])
        except:
            return ' '.join(sentences[:self.num_sentences])

print("Генерация предсказаний Baseline...")
baseline = TFIDFBaseline()

baseline_predictions = []
baseline_references = []

test_data = dataset['test']
sample_size = min(100, len(test_data))

for i in tqdm(range(sample_size), desc="Baseline инференс"):
    pred = baseline.summarize(test_data[i]['review'])
    baseline_predictions.append(pred)
    baseline_references.append(test_data[i]['summary'])

baseline_metrics = metrics_calculator.calculate_all(
    baseline_predictions,
    baseline_references,
    model_name="Baseline TF-IDF"
)

print(f"\n{'='*50}")
print("BASELINE TF-IDF РЕЗУЛЬТАТЫ:")
print(f"{'='*50}")
print(f"  ROUGE-1:           {baseline_metrics['rouge1']:.4f}")
print(f"  ROUGE-2:           {baseline_metrics['rouge2']:.4f}")
print(f"  ROUGE-L:           {baseline_metrics['rougeL']:.4f}")
print(f"  BERTScore F1:      {baseline_metrics['bert_score_f1']:.4f}")
print(f"  SBERT Similarity:  {baseline_metrics['sbert_similarity_mean']:.4f}")


BASELINE TF-IDF МОДЕЛЬ
Генерация предсказаний Baseline...


Baseline инференс: 100%|██████████| 100/100 [00:00<00:00, 174.49it/s]



  Расчет метрик для Baseline TF-IDF...
    ROUGE...
    BERTScore...
    SBERT Semantic Similarity...

BASELINE TF-IDF РЕЗУЛЬТАТЫ:
  ROUGE-1:           0.0331
  ROUGE-2:           0.0029
  ROUGE-L:           0.0291
  BERTScore F1:      0.6233
  SBERT Similarity:  0.5612


In [ ]:
ds = load_dataset("Auttar/KinopoiskReviewsSummarization")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['review', 'summary'],
        num_rows: 16360
    })
    validation: Dataset({
        features: ['review', 'summary'],
        num_rows: 2045
    })
    test: Dataset({
        features: ['review', 'summary'],
        num_rows: 2045
    })
})


## Universal Train_Eval_Push

Следующие блоки будут универсальными

In [ ]:
def train_eval_push(model_id, hub_name, lr=3e-5, epochs=3, prefix=''):
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

    def preprocess(examples):
        inputs = [prefix + t for t in examples['review']]
        model_inputs = tok(inputs, max_length=512, truncation=True, padding='max_length')
        labels = tok(examples['summary'], max_length=128, truncation=True, padding='max_length')
        model_inputs['labels'] = [[-100 if t==tok.pad_token_id else t for t in seq] for seq in labels['input_ids']]
        return model_inputs

    train_ds = ds['train'].map(preprocess, batched=True, remove_columns=ds['train'].column_names)
    val_ds = ds['validation'].map(preprocess, batched=True, remove_columns=ds['validation'].column_names)

    args = Seq2SeqTrainingArguments(
        output_dir='tmp_out',
        eval_strategy='epoch',
        save_strategy='epoch',
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        fp16=True,
        predict_with_generate=True,
        generation_max_length=128,
        generation_num_beams=3,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        push_to_hub=True,
        hub_model_id=hub_name,
        hub_strategy='every_save',
        report_to='none'
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tok, model=model)
    )

    print(f'Обучение {hub_name}...')
    trainer.train()
    trainer.push_to_hub(f'{hub_name} ready')



## Train RuBart

In [ ]:
bart_tr, bart_sc, bart_pred, bart_ref = train_eval_push(
    'sn4kebyt3/ru-bart-large', 'Auttar/kinopoisk-rubart', lr=3e-5, epochs=3)

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

🚀 Обучение Auttar/kinopoisk-rubart...


Epoch,Training Loss,Validation Loss
1,5.713747,1.341441
2,4.557049,1.291607
3,3.693579,1.315905


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  ...tmp_out/training_args.bin: 100%|##########| 5.33kB / 5.33kB            

  ...tmp_out/model.safetensors:  11%|#         |  160MB / 1.52GB            

Инференс: 100%|██████████| 100/100 [02:18<00:00,  1.38s/it]



  Расчет метрик для Auttar/kinopoisk-rubart...
    ROUGE...
    BERTScore...
    SBERT Semantic Similarity...


TypeError: Object of type float32 is not JSON serializable

## Convert Data

In [ ]:
import numpy as np

def _to_py(obj):
    """рекурсивно конвертирует np-скаляры и массивы в обычные типы Python"""
    if isinstance(obj, dict):
        return {k: _to_py(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_py(i) for i in obj]
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


In [ ]:
!pip install jsonlines

In [ ]:
import json, numpy as np
class NpEncoder(json.JSONEncoder):                       # <-- конвертер
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        return super().default(obj)


## Test Generation

In [ ]:
MODEL_ID = "Auttar/kinopoisk-rubart" # Можно выбрать нужную модель
DATASET  = "Auttar/KinopoiskReviewsSummarization"
N_FIRST  = 100
OUT_FILE = "first100_full.json"

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).to(device).eval()

ds = load_dataset(DATASET, split="test").select(range(N_FIRST))

preds, refs = [], []
for sample in tqdm(ds, desc="Generate"):
    txt = sample["review"]
    inp = tok(txt, return_tensors="pt", max_length=512, truncation=True).to(device)
    with torch.no_grad():
        out_ids = model.generate(
            **inp, max_length=128, num_beams=3,
            early_stopping=True, no_repeat_ngram_size=3
        )
    preds.append(tok.decode(out_ids[0], skip_special_tokens=True))
    refs.append(sample["summary"])



Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Generate: 100%|██████████| 100/100 [02:05<00:00,  1.25s/it]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Метрики инициализированы
  Расчет ROUGE...
  Расчет семантической близости...
{
  "rouge1": 0.057999999999999996,
  "rouge2": 0.026666666666666665,
  "rougeL": 0.057999999999999996,
  "semantic_similarity_mean": 0.7540450096130371,
  "semantic_similarity_std": 0.09500998258590698,
  "semantic_similarity_min": 0.5227270126342773,
  "semantic_similarity_max": 0.9173068404197693
}
✅ Готово, результат в first100_full.json


## Calculate metrix and save

In [ ]:
calc = FullMetricsCalculator()
scores = calc.calculate_all(preds, refs)

with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump({"preds": preds, "refs": refs, "scores": scores},
              f, cls=NpEncoder, ensure_ascii=False, indent=2)   # <-- cls=
print(json.dumps(scores, indent=2, ensure_ascii=False, cls=NpEncoder))
print(f"Готово, результат в {OUT_FILE}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Все метрики инициализированы

  Расчет метрик для Model...
    ROUGE...
    BERTScore...
    SBERT Semantic Similarity...
{
  "rouge1": 0.057999999999999996,
  "rouge2": 0.026666666666666665,
  "rougeL": 0.057999999999999996,
  "bert_score_precision": 0.741934597492218,
  "bert_score_recall": 0.7466683387756348,
  "bert_score_f1": 0.7441781759262085,
  "sbert_similarity_mean": 0.7540450096130371,
  "sbert_similarity_std": 0.09500998258590698,
  "sbert_similarity_min": 0.5227270126342773,
  "sbert_similarity_max": 0.9173068404197693
}
✅ Готово, результат в first100_full.json


## Upload Dataset (Corrected)

In [ ]:
!pip install huggingface_hub datasets -q

from huggingface_hub import login
from datasets import Dataset, DatasetDict, load_dataset
import pandas as pd
import json

print("Войдите в Hugging Face:")
login()

Войдите в Hugging Face:


In [ ]:
import json
import pandas as pd

print("Загрузка JSON файлов...")

def load_json_with_fallback(filename):
    encodings = ['utf-8', 'latin-1', 'cp1251', 'utf-16', 'utf-8-sig']

    for encoding in encodings:
        try:
            with open(filename, 'r', encoding=encoding) as f:
                data = json.load(f)
            print(f"{filename} загружен с кодировкой {encoding}")
            return data
        except UnicodeDecodeError:
            continue
        except json.JSONDecodeError as e:
            print(f"{filename}: JSON ошибка - {e}")
            try:
                data = []
                with open(filename, 'r', encoding=encoding) as f:
                    for line in f:
                        if line.strip():
                            data.append(json.loads(line))
                print(f"{filename} загружен как JSON Lines с кодировкой {encoding}")
                return data
            except:
                continue
        except Exception as e:
            print(f"{filename}: {e}")
            continue

    return None

train_data = load_json_with_fallback('train.json')
val_data = load_json_with_fallback('validation.json')
test_data = load_json_with_fallback('test.json')

if train_data is None:
    print("\nНе удалось загрузить JSON файлы.")

print(f"\nTrain: {len(train_data)} записей" if train_data else "Train: None")
print(f"Validation: {len(val_data)} записей" if val_data else "Validation: None")
print(f"Test: {len(test_data)} записей" if test_data else "Test: None")

Загрузка JSON файлов...
  ✓ train.json загружен с кодировкой utf-8
  ✓ validation.json загружен с кодировкой utf-8
  ✓ test.json загружен с кодировкой utf-8

Train: 16360 записей
Validation: 2045 записей
Test: 2045 записей


In [ ]:
print("\nПреобразование в DatasetDict...")

if isinstance(train_data, list):
    train_dataset = Dataset.from_list(train_data)
    val_dataset = Dataset.from_list(val_data)
    test_dataset = Dataset.from_list(test_data)

elif isinstance(train_data, pd.DataFrame):
    train_dataset = Dataset.from_pandas(train_data)
    val_dataset = Dataset.from_pandas(val_data)
    test_dataset = Dataset.from_pandas(test_data)

else:
    train_dataset = Dataset.from_dict({
        'review': [item.get('review', item.get('text', '')) for item in train_data],
        'summary': [item.get('summary', item.get('resume', '')) for item in train_data]
    })
    val_dataset = Dataset.from_dict({
        'review': [item.get('review', item.get('text', '')) for item in val_data],
        'summary': [item.get('summary', item.get('resume', '')) for item in val_data]
    })
    test_dataset = Dataset.from_dict({
        'review': [item.get('review', item.get('text', '')) for item in test_data],
        'summary': [item.get('summary', item.get('resume', '')) for item in test_data]
    })

dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

print(f"DatasetDict создан!")
print(f"Train: {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"Test: {len(dataset['test'])}")


Преобразование в DatasetDict...
✅ DatasetDict создан!
Train: 16360
Validation: 2045
Test: 2045


In [ ]:
print(f"Колонки: {dataset['train'].column_names}")

print(f"\nПример train[0]:")
print(f"  review: {dataset['train'][0]['review'][:100]}...")
print(f"  summary: {dataset['train'][0]['summary'][:100]}...")

print(f"\nПример validation[0]:")
print(f"  review: {dataset['validation'][0]['review'][:100]}...")
print(f"  summary: {dataset['validation'][0]['summary'][:100]}...")

print(f"\nПример test[0]:")
print(f"  review: {dataset['test'][0]['review'][:100]}...")
print(f"  summary: {dataset['test'][0]['summary'][:100]}...")


Проверка данных:
Колонки: ['review', 'summary']

Пример train[0]:
  review: Мне первая часть «Не вижу зла» нравится до сих пор. Уж больно необычный был там маньяк, и он довольн...
  summary: **Резюме:**  
**Главная мысль:** Отличительной чертой первого фильма является уникальный персонаж ма...

Пример validation[0]:
  review: Увидев однажды афишу этого фильма в кинотеатре, я подумал 'Фильмов про вампиров пруд пруди, толковых...
  summary: Резюме отзыва:

Фильм "Другой мир" начинается с потрясающей атмосферы нью-готики и мрачного дождливо...

Пример test[0]:
  review: Помимо очевидных достоинств, о которых упоминают все, хочется выделить безупречное воспроизведение п...
  summary: Резюме:

Фильм создает уникальный атмосферный контекст 1960-х, воплощающий стиль, актерскую игру и с...


In [ ]:
dataset_name = "Auttar/KinopoiskReviewsSummarization"

dataset.push_to_hub(
    dataset_name,
    private=False,
    config_name="default",
    commit_message="Full dataset with train/validation/test splits"
)

print(f"\n✅ Датасет опубликован!")
print(f"📁 Ссылка: https://huggingface.co/datasets/{dataset_name}")


ПУБЛИКАЦИЯ ДАТАСЕТА


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/17 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  77%|#######7  | 3.68MB / 4.76MB            


✅ Датасет опубликован!
📁 Ссылка: https://huggingface.co/datasets/Auttar/KinopoiskReviewsSummarization


## Check Dataset

In [ ]:
!rm -rf ~/.cache/huggingface/datasets

test_load = load_dataset(dataset_name)

print(f"\nПроверка загрузки:")
print(f"Train: {len(test_load['train'])}")
print(f"Validation: {len(test_load['validation'])}")
print(f"Test: {len(test_load['test'])}")

print(f"\nПример из загруженного датасета:")
print(f"review: {test_load['train'][0]['review'][:100]}...")
print(f"summary: {test_load['train'][0]['summary'][:100]}...")

print("\nВСЕ РАБОТАЕТ! Датасет загружен корректно!")


Проверка загруженного датасета...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/38.4M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/4.79M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/4.76M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16360 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2045 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2045 [00:00<?, ? examples/s]


Проверка загрузки:
Train: 16360
Validation: 2045
Test: 2045

Пример из загруженного датасета:
review: Мне первая часть «Не вижу зла» нравится до сих пор. Уж больно необычный был там маньяк, и он довольн...
summary: **Резюме:**  
**Главная мысль:** Отличительной чертой первого фильма является уникальный персонаж ма...

✅ ВСЕ РАБОТАЕТ! Датасет загружен корректно!


##Creating README


In [ ]:
pip install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 31.2 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.4.2
    Uninstalling hf-xet-1.4.2:
      Successfully uninstalled hf-xet-1.4.2
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0


In [ ]:
baseline_rouge1 = baseline_metrics['rouge1']
baseline_rouge2 = baseline_metrics['rouge2']
baseline_rougeL = baseline_metrics['rougeL']
baseline_bert = baseline_metrics['bert_score_f1']
baseline_sbert = baseline_metrics['sbert_similarity_mean']

In [ ]:
import json
from huggingface_hub import HfApi, login

REPO_ID = "Auttar/kinopoisk-rubart" # заменить на необходимый
RESULT_FILE = "first100_full.json"  # файл с метриками

login()

api = HfApi()
api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True
)
print(f"Репозиторий {REPO_ID} готов")

try:
    with open(RESULT_FILE, encoding="utf-8") as f:
        data = json.load(f)

        print(f"Ключи в файле: {data.keys()}")

        scores = data.get('scores', data)

        rut5_rouge1 = scores.get('rouge1', scores.get('rut5_rouge1'))
        rut5_rouge2 = scores.get('rouge2', scores.get('rut5_rouge2'))
        rut5_rougeL = scores.get('rougeL', scores.get('rut5_rougeL'))

        rut5_bert = scores.get('bert_score_f1', scores.get('bert_f1'))

        rut5_sbert = scores.get('sbert_similarity_mean', scores.get('sbert_mean'))

        print(f"Метрики загружены из {RESULT_FILE}")
        print(f"\nruT5-base:")
        print(f"   ROUGE-1: {rut5_rouge1:.4f}")
        print(f"   ROUGE-2: {rut5_rouge2:.4f}")
        print(f"   ROUGE-L: {rut5_rougeL:.4f}")
        print(f"   BERTScore: {rut5_bert:.4f}")
        print(f"   SBERT Sim: {rut5_sbert:.4f}")

except Exception as e:
    print(f"Ошибка чтения {RESULT_FILE}: {e}")

readme_content = f"""---
language: ru
license: mit
tags:
- summarization
- bart
- russian
- sentiment-analysis
pipeline_tag: summarization
---

# ruBART for Kinopoisk Review Summarization

## Описание
Модель для генеративной суммаризации отзывов на русском языке. Обучалась на датасете Kinopoisk с использованием дистилляции знаний от LLM (Qwen/Qwen2.5-1.5B-Instruct).

## Задача
Создание кратких, информативных резюме отзывов с выделением ключевых плюсов и минусов.

## Метрики на тестовой выборке

| Модель | ROUGE-1 | ROUGE-2 | ROUGE-L | BERTScore | SBERT Sim |
|--------|---------|---------|---------|-----------|-----------|
| Baseline (TF-IDF) | {baseline_rouge1:.4f} | {baseline_rouge2:.4f} | {baseline_rougeL:.4f} | {baseline_bert:.4f} | {baseline_sbert:.4f} |
| **ruBART** | **{rut5_rouge1:.4f}** | **{rut5_rouge2:.4f}** | **{rut5_rougeL:.4f}** | **{rut5_bert:.4f}** | **{rut5_sbert:.4f}** |

## Улучшение относительно Baseline

| Метрика | Улучшение |
|--------|-----------|
| ROUGE-1 | +{(rut5_rouge1 - baseline_rouge1)*100:.1f}% |
| ROUGE-2 | +{(rut5_rouge2 - baseline_rouge2)*100:.1f}% |
| ROUGE-L | +{(rut5_rougeL - baseline_rougeL)*100:.1f}% |
| BERTScore | +{(rut5_bert - baseline_bert)*100:.1f}% |
| SBERT Sim | +{(rut5_sbert - baseline_sbert)*100:.1f}% |

## Использование

```python
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("Auttar/kinopoisk-rubart")
tokenizer = AutoTokenizer.from_pretrained("Auttar/kinopoisk-rubart")

def summarize(review):
    input_text = f"summarize: {{review}}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(inputs.input_ids, max_length=128, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Пример
review = "Фильм просто бомба! Актеры играют отлично, сюжет держит в напряжении."
print(summarize(review))

`````

# Обучение
Базовая модель: sn4kebyt3/ru-bart-large

Датасет: Auttar/KinopoiskReviewsSummarization (18,405 пар)

Эпохи: 3

Learning rate: 3e-5

Batch size: 4

Оптимизатор: AdamW

# Ссылки
Датасет: https://huggingface.co/datasets/Auttar/KinopoiskReviewsSummarization

Модель: https://huggingface.co/Auttar/kinopoisk-rubart
"""

ЗАГРУЗКА README НА HUGGING FACE HUB
✅ Метрики загружены из first100_full.json


## Upload to HF

In [ ]:
try:
  api = HfApi()

  api.upload_file(
  path_or_fileobj=readme_content.encode('utf-8'),
  path_in_repo="README.md",
  repo_id=REPO_ID,
  repo_type="model"
  )

  print(f"\nREADME успешно загружен!")
  print(f"🔗 https://huggingface.co/{REPO_ID}")

except Exception as e:
  print(f"\nОшибка: {e}")
  print("\nНужно войти в Hugging Face аккаунт")

  from huggingface_hub import login
  login()

  api.upload_file(
  path_or_fileobj=readme_content.encode('utf-8'),
  path_in_repo="README.md",
  repo_id=REPO_ID,
  repo_type="model"
  )
  print(f"\nREADME загружен!")



ЗАГРУЗКА README НА HUGGING FACE HUB

✅ README успешно загружен!
🔗 https://huggingface.co/Auttar/kinopoisk-rubart


# Pegasus

## Universal Train_eval_push (Updated for Pegasus)

In [ ]:
def train_eval_push(model_id, hub_name, dataset, lr=3e-5, epochs=3, prefix='', model_type='pegasus'):
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    if model_type == 'pegasus':
        from transformers import PegasusTokenizer, PegasusForConditionalGeneration
        tok = PegasusTokenizer.from_pretrained(model_id)
        model = PegasusForConditionalGeneration.from_pretrained(model_id).to(device)

        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        use_prefix = False

    elif model_type == 't5':
        from transformers import T5Tokenizer, T5ForConditionalGeneration
        tok = T5Tokenizer.from_pretrained(model_id)
        model = T5ForConditionalGeneration.from_pretrained(model_id).to(device)
        use_prefix = True

    elif model_type == 'bart':
        from transformers import BartTokenizer, BartForConditionalGeneration
        tok = BartTokenizer.from_pretrained(model_id)
        model = BartForConditionalGeneration.from_pretrained(model_id).to(device)
        use_prefix = False

    else:
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        tok = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)
        use_prefix = True if 't5' in model_id.lower() else False

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    print(f"Токенизатор и модель загружены: {type(tok).__name__}")

    def preprocess(examples):
        if use_prefix and prefix:
            inputs = [prefix + t for t in examples['review']]
        else:
            inputs = examples['review']
        model_inputs = tok(
            inputs,
            max_length=512,
            truncation=True,
            padding='max_length'
        )
        labels = tok(
            examples['summary'],
            max_length=128,
            truncation=True,
            padding='max_length'
        )

        model_inputs['labels'] = [
            [-100 if t == tok.pad_token_id else t for t in seq]
            for seq in labels['input_ids']
        ]

        return model_inputs

    train_ds = dataset['train'].map(preprocess, batched=True, remove_columns=dataset['train'].column_names)
    val_ds = dataset['validation'].map(preprocess, batched=True, remove_columns=dataset['validation'].column_names)

    print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}")

    args = Seq2SeqTrainingArguments(
        output_dir='tmp_out',
        eval_strategy='epoch',
        save_strategy='epoch',
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        fp16=torch.cuda.is_available(),
        predict_with_generate=True,
        generation_max_length=128,
        generation_num_beams=4,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        push_to_hub=True,
        hub_model_id=hub_name,
        hub_strategy='every_save',
        report_to='none',
        logging_steps=50,
        save_total_limit=2,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tok, model=model)
    )

    print(f'Обучение {hub_name}...')
    trainer.train()

    trainer.push_to_hub(commit_message=f'{hub_name} ready')

## Import Dataset (Again)

In [ ]:
from datasets import load_dataset

print("Загрузка датасета...")
ds = load_dataset("Auttar/KinopoiskReviewsSummarization")

print(f"Train: {len(ds['train'])}")
print(f"Validation: {len(ds['validation'])}")
print(f"Test: {len(ds['test'])}")
print(f"\nПример:")
print(f"Отзыв: {ds['train'][0]['review'][:150]}...")
print(f"Резюме: {ds['train'][0]['summary'][:150]}...")

Загрузка датасета...
Train: 16360
Validation: 2045
Test: 2045

Пример:
Отзыв: Мне первая часть «Не вижу зла» нравится до сих пор. Уж больно необычный был там маньяк, и он довольно круто расправлялся со своими жертвами. Фильм как...
Резюме: **Резюме:**  
**Главная мысль:** Отличительной чертой первого фильма является уникальный персонаж маньяка и оригинальное поведение автора. Следующий ф...


## Train Pegasus

In [ ]:
pegasus_tr, pegasus_sc, pegasus_pred, pegasus_ref = train_eval_push(
    model_id='google/pegasus-x-base',
    hub_name='Auttar/kinopoisk-pegasus',
    dataset=ds,
    lr=3e-5,
    epochs=3,
    prefix='',
    model_type='pegasus'
)

You are using a model of type pegasus_x to instantiate a model of type pegasus. This is not supported for all configurations of models and can yield errors.


Loading weights:   0%|          | 0/368 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-x-base
Key                                                              | Status     | 
--------------------------------

✅ Токенизатор и модель загружены: PegasusTokenizer
Подготовка данных...


Map:   0%|          | 0/16360 [00:00<?, ? examples/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tmp_out/training_args.bin: 100%|##########| 5.33kB / 5.33kB            

  ...tmp_out/model.safetensors:   2%|1         | 39.9MB / 2.08GB            

Оценка модели...
